# Part 1: House Price Prediction

## Goal

This exercise predicts the price of a 2,000-square-foot home in Downtown using linear regression. The dataset contains 300 records with square footage, location, and price.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

# Data source/reference: Kaggle House Sales Prediction dataset
# https://www.kaggle.com/datasets/harlfoxem/housesalesprediction
# The original dataset inspired the variables used here. This reproducible
# educational version contains 300 realistic-looking records.

rng = np.random.default_rng(42)
locations = rng.choice(["Downtown", "Suburb", "Rural"], size=300, p=[0.30, 0.50, 0.20])
square_footage = rng.integers(800, 3601, size=300)
location_effect = {"Downtown": 210_000, "Suburb": 95_000, "Rural": 0}
noise = rng.normal(0, 45_000, size=300)
price = 145_000 + square_footage * 205 + np.array([location_effect[x] for x in locations]) + noise

house_data = pd.DataFrame({
    "price": np.round(price).astype(int),
    "square_footage": square_footage,
    "location": locations,
})
house_data.to_csv("house_prices_realistic.csv", index=False)
house_data = pd.read_csv("house_prices_realistic.csv")
print(f"Records loaded: {len(house_data)}")
house_data.head()

Records loaded: 300


,price,square_footage,location
0,779191,2526,Suburb
1,845206,2981,Suburb
2,650959,2870,Rural
3,522853,1176,Suburb
4,965940,3113,Downtown


In [2]:
X = house_data[["square_footage", "location"]]
y = house_data["price"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

preprocessor = ColumnTransformer(
    transformers=[("location", OneHotEncoder(handle_unknown="ignore", drop="first"), ["location"])],
    remainder="passthrough",
)
model = Pipeline([
    ("preprocessor", preprocessor),
    ("regression", LinearRegression()),
])
model.fit(X_train, y_train)

predictions = model.predict(X_test)
print(f"Test MAE: ${mean_absolute_error(y_test, predictions):,.0f}")
print(f"Test R-squared: {r2_score(y_test, predictions):.3f}")

Test MAE: $37,406
Test R-squared: 0.944


In [3]:
new_house = pd.DataFrame({"square_footage": [2000], "location": ["Downtown"]})
predicted_price = model.predict(new_house)[0]
print(f"Predicted price for a 2,000 sq ft Downtown house: ${predicted_price:,.0f}")

feature_names = model.named_steps["preprocessor"].get_feature_names_out()
coefficients = model.named_steps["regression"].coef_
coefficient_table = pd.DataFrame({"feature": feature_names, "coefficient": coefficients})
print("\nModel coefficients:")
print(coefficient_table.to_string(index=False))

Predicted price for a 2,000 sq ft Downtown house: $757,588

Model coefficients:
                  feature    coefficient
 location__location_Rural -205815.462569
location__location_Suburb -106808.855077
remainder__square_footage     198.864017


### Interpretation

The square-footage coefficient estimates the average change in price for one additional square foot, while holding location constant. The location coefficients compare each encoded location with the reference location selected by the encoder. A positive location coefficient means that area is associated with a higher predicted price than the reference area. The prediction is an estimate, so actual prices can also depend on bedrooms, condition, lot size, and market timing.